# 🗣️ Módulo 12 — Orchestration: Group Chat

> **Objetivo:** Coordinar una conversación grupal entre varios agentes con un orquestador que decide quién habla a continuación.

📚 **Doc oficial:** [Group Chat Orchestration (Python)](https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/group-chat?pivots=programming-language-python)

## ¿Qué es Group Chat Orchestration?

Topología en **estrella** — todos los agentes hablan a través de un orquestador central que
decide qué agente toma el turno siguiente, basándose en una función de selección o en otro agente.

```
              ┌───────────┐
              │Orchestrator│
              └─┬───┬───┬─┘
        ┌───────┘   │   └────────┐
   ┌────▼───┐   ┌───▼───┐   ┌───▼────┐
   │ Agent A│   │Agent B│   │ Agent C│
   └────────┘   └───────┘   └────────┘
```

Ideal para:
- **Refinamiento iterativo** — round-robin de revisores
- **Colaboración multi-perspectiva** — Researcher + Writer + Critic
- Debates / brainstorming estructurado

## Selectores disponibles

| Selector | API | Cuándo usar |
|----------|-----|-------------|
| Round-robin | `selection_func=round_robin_selector` | Todos hablan por turno |
| Custom Python | `selection_func=lambda state: ...` | Lógica determinista basada en estado |
| Agent-based | `orchestrator_agent=Agent(...)` | LLM decide quién habla (con contexto) |


## ⚙️ Setup

In [1]:
# Aseguramos que la raíz del proyecto esté en sys.path para importar `helpers`
import sys
from pathlib import Path

_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from agent_framework import Agent
from helpers.config import create_chat_client
print("✅ Helpers cargados.")


C:\Users\jreyesleger\AppData\Roaming\Python\Python312\site-packages\agent_framework\_skills.py:116: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
C:\Users\jreyesleger\AppData\Roaming\Python\Python312\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


✅ Helpers cargados.


## 1️⃣ Round-robin selector

El selector más simple — recorre los participantes en orden. Útil cuando todos deben
contribuir por igual.


In [ ]:
from agent_framework.orchestrations import GroupChatBuilder, GroupChatState

client = create_chat_client()

researcher = Agent(
    client=client,
    name="Researcher",
    description="Recopila información relevante de fondo.",
    instructions="Recopila datos concisos que ayuden a responder la pregunta. Sé breve y factual. Responde siempre en español.",
    require_per_service_call_history_persistence=True,
)

writer = Agent(
    client=client,
    name="Writer",
    description="Sintetiza respuestas pulidas usando la información recopilada.",
    instructions="Compón respuestas claras y estructuradas usando las notas proporcionadas. Sé completo. Responde siempre en español.",
    require_per_service_call_history_persistence=True,
)


def round_robin_selector(state: GroupChatState) -> str:
    """Selecciona al siguiente speaker en round-robin."""
    names = list(state.participants.keys())
    return names[state.current_round % len(names)]


workflow = GroupChatBuilder(
    participants=[researcher, writer],
    termination_condition=lambda conv: sum(1 for m in conv if m.role == "assistant") >= 4,
    selection_func=round_robin_selector,
    intermediate_output_from=[researcher, writer],
).build()

print("✅ Group chat configurado con round-robin (máx 4 mensajes de agentes)")

✅ Group chat configurado con round-robin (máx 4 mensajes de agentes)


## 2️⃣ Ejecutar y observar la conversación

Activa `stream=True` para ver cada actualización a medida que llega.
El output terminal es un `AgentResponse` con el resumen del orquestador.


In [ ]:
from agent_framework import AgentResponse, AgentResponseUpdate

task = "¿Cuáles son los beneficios clave de async/await en Python?"

print(f"📨 Task: {task}\n{'=' * 60}\n")

last_author = None

async for event in workflow.run(task, stream=True):
    data = event.data
    if isinstance(data, AgentResponseUpdate) and data.text:
        if data.author_name != last_author:
            if last_author is not None:
                print("\n")
            print(f"🤖 [{data.author_name}]: ", end="", flush=True)
            last_author = data.author_name
        print(data.text, end="", flush=True)

print(f"\n\n{'=' * 60}\n✅ Conversación completada.")

📨 Task: What are the key benefits of async/await in Python?

🤖 [Researcher]: 1. **Improved Readability**: Async/await syntax makes asynchronous code easier to write, read, and understand, resembling synchronous code structure.

2. **Concurrency**: Allows better handling of I/O-bound tasks by enabling non-blocking code execution, improving performance in tasks like web requests or file handling.

3. **Efficient Resource Use**: Prevents blocking threads during I/O operations, enabling efficient use of system resources like CPU and memory.

4. **Scalability**: Supports handling a large number of simultaneous tasks (e.g., multiple client connections) without needing many threads.

5. **Error Handling**: Errors in async/await code are easier to catch and debug compared to callback-based asynchronous programming.

6. **Interoperability**: Works seamlessly with Python's `asyncio` framework, enabling integration with event loops, coroutines, and asynchronous libraries. 

7. **Simplifies Callba

## 3️⃣ Selector custom basado en contenido

`GroupChatState.conversation` te da acceso a todos los mensajes hasta ahora.
Puedes implementar lógica inteligente — por ejemplo, que el `Researcher` investigue
primero y luego el `Writer` sintetice la respuesta final.

In [ ]:
def content_based_selector(state: GroupChatState) -> str:
    """Researcher habla primero, Writer sintetiza después. Alterna si Researcher ya contribuyó."""
    assistant_msgs = [m for m in state.conversation if m.role == "assistant"]

    if len(assistant_msgs) == 0:
        return "Researcher"  # Primera ronda: investigar

    last = assistant_msgs[-1]
    # Después de que Researcher habla, Writer sintetiza
    if last.author_name == "Researcher":
        return "Writer"
    # Después de Writer, Researcher puede añadir más datos
    return "Researcher"


workflow = GroupChatBuilder(
    participants=[researcher, writer],
    termination_condition=lambda conv: sum(1 for m in conv if m.role == "assistant") >= 4,
    selection_func=content_based_selector,
    intermediate_output_from=[researcher, writer],
).build()

print("📨 Task: Explicar los transformadores de mónadas\n")

last_author = None
async for event in workflow.run("Explica brevemente qué son los transformadores de mónadas en programación funcional.", stream=True):
    data = event.data
    if isinstance(data, AgentResponseUpdate) and data.text:
        if data.author_name != last_author:
            if last_author is not None:
                print("\n")
            print(f"🤖 [{data.author_name}]: ", end="", flush=True)
            last_author = data.author_name
        print(data.text, end="", flush=True)

print("\n\n✅ Selector basado en contenido completado.")

📨 Task: Explain monad transformers

🤖 [Researcher]: Monad transformers are structures in functional programming (specifically in Haskell) that allow combining multiple monads into a single, cohesive monadic context. This avoids the problem of "monad stacking," where multiple nested layers of monads make code unwieldy. 

For example:
- A `MaybeT` transformer combines the effects of the `Maybe` monad with another monad (e.g., `IO`).
- They provide a way to "lift" actions from the inner monad to the combined monad.

Monad transformers simplify composing and managing multiple computational effects in a cleaner, modular way.

🤖 [Writer]: Monad transformers are constructs used in functional programming, especially in Haskell, to combine and work with multiple monadic contexts in a composable and manageable way. They allow you to stack monads to handle multiple effects (e.g., state, I/O, errors, etc.) without deeply nesting monadic operations.

### Key Points:
1. **Motivation**: When you need

## 4️⃣ Orquestador basado en agente (LLM)

Para conversaciones más complejas, deja que **otro agente** decida quién habla a continuación.
El orquestador es un `Agent` completo con sus propias instructions y tools.


In [ ]:
orchestrator_agent = Agent(
    client=client,
    name="Orchestrator",
    description="Coordina la colaboración multi-agente seleccionando quién habla",
    instructions=(
        "Coordinas una conversación de equipo para resolver la tarea del usuario.\n\n"
        "Directrices:\n"
        "- Empieza con Researcher para recopilar información\n"
        "- Luego Writer sintetiza la respuesta final\n"
        "- Alterna entre ellos — nunca selecciones al mismo agente dos veces seguidas\n"
        "- Solo termina cuando ambos hayan contribuido significativamente\n"
        "- Responde siempre en español"
    ),
    require_per_service_call_history_persistence=True,
)

workflow = GroupChatBuilder(
    participants=[researcher, writer],
    termination_condition=lambda msgs: sum(1 for m in msgs if m.role == "assistant") >= 4,
    orchestrator_agent=orchestrator_agent,
    intermediate_output_from=[researcher, writer],
).build()

print("📨 Task: SQLite vs PostgreSQL para una startup SaaS\n")

turns: list[tuple[str, str]] = []
current_author = None
current_text = []

async for event in workflow.run(
    "¿Cuáles son las ventajas y desventajas de usar SQLite vs PostgreSQL para una startup SaaS?",
    stream=True,
):
    data = event.data
    if isinstance(data, AgentResponseUpdate) and data.text:
        if data.author_name != current_author:
            if current_author and current_text:
                turns.append((current_author, "".join(current_text)))
            current_author = data.author_name
            current_text = []
        current_text.append(data.text)

if current_author and current_text:
    turns.append((current_author, "".join(current_text)))

for i, (author, text) in enumerate(turns, 1):
    preview = text[:200] + "..." if len(text) > 200 else text
    print(f"🤖 Turno {i} [{author}]: {preview}\n")

print(f"✅ Conversación completada — {len(turns)} turnos de agentes.")

📨 Task: SQLite vs PostgreSQL for SaaS

🤖 Turno 1 [Researcher]: ### Trade-offs of using SQLite vs. PostgreSQL:

#### SQLite:
**Pros:**
- **Simplicity:** Easy to set up, no server management required.
- **Lightweight:** Minimal resource usage; ideal for small-scale...

🤖 Turno 2 [Writer]: **Trade-offs of Using SQLite vs. PostgreSQL for a SaaS Startup**

When deciding between **SQLite** and **PostgreSQL** for a SaaS startup, it’s important to weigh the benefits and drawbacks of each in ...

🤖 Turno 3 [Orchestrator]: The comprehensive answer has been provided. Please let us know if you need further assistance.

✅ Conversación completada — 3 turnos de agentes.


## 🎯 Resumen

| Capacidad | API |
|-----------|-----|
| Round-robin | `selection_func=round_robin_selector` (con `GroupChatState`) |
| Selector custom | `selection_func=lambda state: "AgentName"` |
| Orquestador LLM | `orchestrator_agent=Agent(...)` |
| Condición de terminación | `termination_condition=lambda conv: len(conv) >= N` |
| Outputs intermedios | `intermediate_output_from=[a, b]` |
| Streaming | `async for event in workflow.run(task, stream=True):` |

## 📊 Comparación de patrones de orquestación

| Patrón | Cuándo usar |
|--------|-------------|
| **Sequential** (mod 09) | Pipeline lineal con etapas claras |
| **Concurrent** (mod 10) | Múltiples perspectivas en paralelo |
| **Handoff** (mod 11) | Delegación dinámica entre especialistas |
| **Group Chat** (mod 12) | Conversación colaborativa con turnos |

**Siguiente módulo →** [Módulo 13 — Workflows: Ejecutores (low-level)](./13_workflows_executors.ipynb)
